# Grid Search and Consensus Solutions Demo

This notebook demonstrates Boomer-Py's grid search capabilities for finding robust ontology mappings across different parameter configurations.

In [ ]:
import sys
sys.path.append('..')

from boomer.model import (
    KB, PFact, EquivalentTo, ProperSubClassOf,
    MemberOfDisjointGroup, SearchConfig, GridSearch
)
from boomer.search import grid_search, solve
from boomer.evaluator import evaluate_facts
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np

## 1. Create Test Knowledge Base

We'll create a KB with competing mappings of varying confidence levels.

In [ ]:
# Create KB with uncertain mappings
kb = KB(
    facts=[
        # Prevent invalid within-ontology equivalences
        MemberOfDisjointGroup(sub="human_heart", group="Human"),
        MemberOfDisjointGroup(sub="human_liver", group="Human"),
        MemberOfDisjointGroup(sub="human_brain", group="Human"),
        MemberOfDisjointGroup(sub="mouse_heart", group="Mouse"),
        MemberOfDisjointGroup(sub="mouse_liver", group="Mouse"),
        MemberOfDisjointGroup(sub="mouse_brain", group="Mouse"),
        ProperSubClassOf(sub="human_heart", sup="organ"),
        ProperSubClassOf(sub="mouse_heart", sup="organ"),
    ],
    pfacts=[
        # High confidence correct mappings
        PFact(fact=EquivalentTo(sub="human_heart", equivalent="mouse_heart"), prob=0.92),
        PFact(fact=EquivalentTo(sub="human_liver", equivalent="mouse_liver"), prob=0.89),
        PFact(fact=EquivalentTo(sub="human_brain", equivalent="mouse_brain"), prob=0.91),
        
        # Medium confidence mappings
        PFact(fact=EquivalentTo(sub="human_heart", equivalent="cardiac_organ"), prob=0.75),
        PFact(fact=EquivalentTo(sub="mouse_heart", equivalent="cardiac_structure"), prob=0.72),
        
        # Low confidence incorrect mappings
        PFact(fact=EquivalentTo(sub="human_heart", equivalent="mouse_liver"), prob=0.15),
        PFact(fact=EquivalentTo(sub="human_brain", equivalent="mouse_liver"), prob=0.08),
        
        # Conflicting mappings
        PFact(fact=EquivalentTo(sub="cardiac_organ", equivalent="cardiac_structure"), prob=0.65),
        PFact(fact=ProperSubClassOf(sub="cardiac_organ", sup="cardiac_structure"), prob=0.35),
    ],
    name="Cross-species Organ Mapping",
)

print(f"KB contains {len(kb.pfacts)} probabilistic facts")
print(f"Search space: 2^{len(kb.pfacts)} = {2**len(kb.pfacts)} possible combinations")

## 2. Define Ground Truth for Evaluation

In [ ]:
# Ground truth based on biological knowledge
ground_truth_kb = KB(
    facts=[
        EquivalentTo(sub="human_heart", equivalent="mouse_heart"),
        EquivalentTo(sub="human_liver", equivalent="mouse_liver"),
        EquivalentTo(sub="human_brain", equivalent="mouse_brain"),
        EquivalentTo(sub="human_heart", equivalent="cardiac_organ"),
    ]
)

print(f"Ground truth contains {len(ground_truth_kb.facts)} correct mappings")

## 3. Configure Grid Search

In [ ]:
# Define parameter grid
grid = GridSearch(
    configurations=[
        SearchConfig(max_iterations=500),
        SearchConfig(max_iterations=2000),
    ],
    configuration_matrix={
        "max_pfacts_per_clique": [5, 10, 20],
        "partition_initial_threshold": [5, 10],
        "timeout_seconds": [10],
    }
)

# Calculate total configurations
n_configs = len(grid.configurations) * 3 * 2 * 1
print(f"Grid search will test {n_configs} configurations")
print(f"Parameters varied:")
print(f"  - max_iterations: [500, 2000]")
print(f"  - max_pfacts_per_clique: [5, 10, 20]")
print(f"  - partition_initial_threshold: [5, 10]")
print(f"  - timeout_seconds: [10]")

## 4. Run Grid Search

In [ ]:
import time

print("Running grid search...")
start = time.time()

results = grid_search(kb, grid, eval_kb=ground_truth_kb)

elapsed = time.time() - start
print(f"\nGrid search completed in {elapsed:.1f} seconds")
print(f"Configurations tested: {len(results.results)}")

## 5. Analyze Aggregate Statistics

In [ ]:
# Display aggregate stats
stats = results.aggregate_stats

if stats:
    metrics_df = pd.DataFrame({
        'Metric': ['Precision', 'Recall', 'F1', 'Confidence', 'Time (s)'],
        'Mean': [
            stats.mean_precision,
            stats.mean_recall,
            stats.mean_f1,
            stats.mean_confidence,
            stats.mean_time
        ],
        'Std Dev': [
            stats.std_precision,
            stats.std_recall,
            stats.std_f1,
            stats.std_confidence,
            stats.std_time
        ]
    })
    
    display(metrics_df)
    
    print(f"\nSuccess rate: {stats.success_rate:.0%}")
    print(f"Timeout rate: {stats.timeout_rate:.0%}")
    print(f"Mean combinations explored: {stats.mean_combinations_explored}")

## 6. Examine Synthesized Consensus Solution

In [ ]:
# Get consensus solution
consensus = results.synthesized_solution

if consensus:
    print(f"Consensus based on {consensus.contributing_configs} configurations")
    print(f"Aggregation method: {consensus.aggregation_method}")
    print(f"\nHigh confidence mappings ({len(consensus.high_confidence_facts)}):")
    for pfact in consensus.high_confidence_facts:
        print(f"  - {pfact.fact}")
    
    print(f"\nUncertain mappings ({len(consensus.uncertain_facts)}):")
    for pfact in consensus.uncertain_facts:
        print(f"  - {pfact.fact}")
    
    # Create detailed consensus DataFrame
    consensus_data = []
    for pc in consensus.pfact_consensus:
        consensus_data.append({
            'Fact': str(pc.pfact.fact),
            'Prior Prob': pc.pfact.prob,
            'Acceptance Rate': pc.acceptance_rate,
            'Mean Posterior': pc.mean_posterior,
            'Consensus Score': pc.consensus_score,
            'Configs Accepting': len(pc.configurations_accepted),
        })
    
    consensus_df = pd.DataFrame(consensus_data)
    consensus_df = consensus_df.sort_values('Consensus Score', ascending=False)
    
    print("\nDetailed Consensus Analysis:")
    display(consensus_df)

## 7. Visualize Results Distribution

In [ ]:
# Extract metrics for visualization
f1_scores = [r.evaluation.f1 for r in results.results if r.evaluation]
times = [r.result.time_elapsed for r in results.results if r.result.time_elapsed]
confidences = [r.result.confidence for r in results.results]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# F1 Score distribution
if f1_scores:
    axes[0].hist(f1_scores, bins=10, edgecolor='black', alpha=0.7)
    axes[0].axvline(np.mean(f1_scores), color='red', linestyle='--', label='Mean')
    axes[0].set_xlabel('F1 Score')
    axes[0].set_ylabel('Count')
    axes[0].set_title('F1 Score Distribution')
    axes[0].legend()

# Execution time distribution
if times:
    axes[1].hist(times, bins=10, edgecolor='black', alpha=0.7, color='green')
    axes[1].axvline(np.mean(times), color='red', linestyle='--', label='Mean')
    axes[1].set_xlabel('Time (seconds)')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Execution Time Distribution')
    axes[1].legend()

# Confidence distribution
axes[2].hist(confidences, bins=10, edgecolor='black', alpha=0.7, color='orange')
axes[2].axvline(np.mean(confidences), color='red', linestyle='--', label='Mean')
axes[2].set_xlabel('Solution Confidence')
axes[2].set_ylabel('Count')
axes[2].set_title('Confidence Distribution')
axes[2].legend()

plt.tight_layout()
plt.show()

## 8. Identify Best Configuration

In [ ]:
if results.best_config:
    print(f"Best configuration (by {results.best_config_metric}):")
    print(f"  Max iterations: {results.best_config.max_iterations}")
    print(f"  Max pfacts per clique: {results.best_config.max_pfacts_per_clique}")
    print(f"  Partition threshold: {results.best_config.partition_initial_threshold}")
    print(f"  Timeout: {results.best_config.timeout_seconds}s")
    
    # Find the result for best config
    best_result = next(r for r in results.results if r.config == results.best_config)
    if best_result.evaluation:
        print(f"\nBest config performance:")
        print(f"  F1: {best_result.evaluation.f1:.3f}")
        print(f"  Precision: {best_result.evaluation.precision:.3f}")
        print(f"  Recall: {best_result.evaluation.recall:.3f}")
    print(f"  Confidence: {best_result.result.confidence:.3f}")
    print(f"  Time: {best_result.result.time_elapsed:.2f}s")

## 9. Analyze Pareto Frontier

In [ ]:
if results.pareto_frontier:
    print(f"Pareto frontier contains {len(results.pareto_frontier)} configurations")
    print("\nPareto optimal configurations (speed vs accuracy trade-off):\n")
    
    pareto_data = []
    for r in results.pareto_frontier:
        pareto_data.append({
            'Max Iterations': r.config.max_iterations,
            'Clique Size': r.config.max_pfacts_per_clique,
            'Time (s)': r.result.time_elapsed,
            'F1': r.evaluation.f1 if r.evaluation else None,
            'Confidence': r.result.confidence,
        })
    
    pareto_df = pd.DataFrame(pareto_data)
    pareto_df = pareto_df.sort_values('Time (s)')
    display(pareto_df)
    
    # Visualize Pareto frontier
    if f1_scores and times:
        plt.figure(figsize=(8, 5))
        
        # All configs
        all_times = [r.result.time_elapsed for r in results.results]
        all_f1s = [r.evaluation.f1 if r.evaluation else 0 for r in results.results]
        plt.scatter(all_times, all_f1s, alpha=0.5, label='All configs')
        
        # Pareto frontier
        pareto_times = [r.result.time_elapsed for r in results.pareto_frontier]
        pareto_f1s = [r.evaluation.f1 if r.evaluation else 0 for r in results.pareto_frontier]
        plt.scatter(pareto_times, pareto_f1s, color='red', s=100, 
                   marker='*', label='Pareto frontier', zorder=5)
        
        # Connect Pareto points
        sorted_indices = np.argsort(pareto_times)
        sorted_times = [pareto_times[i] for i in sorted_indices]
        sorted_f1s = [pareto_f1s[i] for i in sorted_indices]
        plt.plot(sorted_times, sorted_f1s, 'r--', alpha=0.5)
        
        plt.xlabel('Execution Time (seconds)')
        plt.ylabel('F1 Score')
        plt.title('Speed vs Accuracy Trade-off')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

## 10. Parameter Impact Analysis

In [ ]:
# Analyze impact of each parameter
param_impacts = {}

# Max iterations impact
if f1_scores:
    iterations = [r.config.max_iterations for r in results.results if r.evaluation]
    if len(set(iterations)) > 1:
        corr = np.corrcoef(iterations, f1_scores[:len(iterations)])[0,1]
        param_impacts['max_iterations'] = abs(corr)

# Clique size impact
clique_sizes = [r.config.max_pfacts_per_clique for r in results.results if r.evaluation]
if len(set(clique_sizes)) > 1:
    corr = np.corrcoef(clique_sizes, f1_scores[:len(clique_sizes)])[0,1]
    param_impacts['max_pfacts_per_clique'] = abs(corr)

# Partition threshold impact
thresholds = [r.config.partition_initial_threshold for r in results.results if r.evaluation]
if len(set(thresholds)) > 1:
    corr = np.corrcoef(thresholds, f1_scores[:len(thresholds)])[0,1]
    param_impacts['partition_initial_threshold'] = abs(corr)

if param_impacts:
    print("Parameter Impact on F1 Score (absolute correlation):")
    for param, impact in sorted(param_impacts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {param}: {impact:.3f}")
        
    # Visualize parameter impacts
    plt.figure(figsize=(8, 4))
    params = list(param_impacts.keys())
    impacts = list(param_impacts.values())
    plt.bar(params, impacts, color=['red' if i > 0.5 else 'blue' for i in impacts])
    plt.ylabel('Impact (|correlation|)')
    plt.title('Parameter Impact on F1 Score')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 11. Compare Single Config vs Consensus

In [ ]:
# Run a single configuration for comparison
print("Running single configuration for comparison...")
single_config = SearchConfig(
    max_iterations=1000,
    max_pfacts_per_clique=10,
    partition_initial_threshold=10,
    timeout_seconds=10
)

single_solution = solve(kb, single_config)

# Evaluate single solution
single_accepted = [
    spf.pfact.fact for spf in single_solution.solved_pfacts
    if spf.truth_value and spf.posterior_prob > 0.5
]
single_eval = evaluate_facts(ground_truth_kb.facts, single_accepted)

# Compare with consensus
print("\n" + "="*50)
print("Comparison: Single Config vs Grid Search Consensus")
print("="*50)

comparison_df = pd.DataFrame({
    'Approach': ['Single Config', 'Grid Search Best', 'Grid Search Consensus'],
    'F1 Score': [
        single_eval.f1,
        max(r.evaluation.f1 for r in results.results if r.evaluation) if f1_scores else 0,
        stats.mean_f1 if stats else 0
    ],
    'Time (s)': [
        single_solution.time_elapsed,
        elapsed,
        elapsed
    ],
    'Confidence': [
        single_solution.confidence,
        max(r.result.confidence for r in results.results),
        stats.mean_confidence if stats else 0
    ]
})

display(comparison_df)

print("\nKey Insights:")
if stats and single_eval.f1 < stats.mean_f1:
    print("✓ Grid search found better configurations than single attempt")
else:
    print("✗ Single configuration performed well; grid search may be unnecessary")

if consensus and consensus.high_confidence_facts:
    print(f"✓ Consensus identified {len(consensus.high_confidence_facts)} robust mappings")
    print(f"✓ Parameter-sensitive mappings: {len(consensus.uncertain_facts)}")

## Conclusion

This demonstration showed how grid search with consensus solutions:

1. **Explores parameter space systematically** - Testing multiple configurations
2. **Identifies robust mappings** - Via consensus across configurations
3. **Quantifies uncertainty** - Through acceptance rates and variance
4. **Finds optimal trade-offs** - Using Pareto frontier analysis
5. **Provides deployment confidence** - With aggregated statistics

The synthesized solution is more reliable than any single configuration, making it ideal for production use where robustness matters more than perfect optimization.